# Eval Fine-tuned Qwen3-4B vs Gemini on Private Repo

Head-to-head comparison of a **fine-tuned Qwen3-4B** (served via vLLM) against **Gemini 3 Flash** (via API) on a private repo.

**Pipeline:**
1. Collect & sample files from your private repo in `repos/`
2. Run both models on the same files in parallel
3. Compute agreement metrics (F1 / recall / precision)

**Prerequisites:**
- `pip install -r requirements-eval.txt`
- vLLM serving your fine-tuned model:
  ```bash
  vllm serve /path/to/finetuned-qwen3-4b --max-model-len 16384
  ```
- Gemini API key (Google AI Studio or OpenRouter)
- Place your private repo in `repos/<repo_name>/`

In [ ]:
%pip install -q -r requirements-eval.txt

In [ ]:
import os, sys, json, time, random
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

from lib.collector import collect_files
from lib.llm_client import llm_extract
from lib.languages import get_language_config

print(f"Root: {ROOT}")

## Config

Edit the cells below to match your setup.

In [ ]:
# ── Private repo settings ──
REPO_NAME     = "my-private-repo"          # folder name inside repos/
REPO_LANGUAGE = "python"                   # python | java | csharp | cpp | javascript

# ── Fine-tuned Qwen model (served via vLLM) ──
VLLM_URL      = "http://localhost:8000/v1"
QWEN_MODEL_ID = "finetuned-qwen3-4b"      # model name as loaded in vLLM
QWEN_WORKERS  = 4
QWEN_MAX_TOKENS = 1024
QWEN_EXTRA_BODY = {"chat_template_kwargs": {"enable_thinking": False}}

# ── Gemini (teacher) ──
# Option A: Google AI Studio (direct)
GEMINI_URL    = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL  = "gemini-2.5-flash"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")  # export GEMINI_API_KEY=...

# Option B: OpenRouter (uncomment below, comment out Option A)
# GEMINI_URL    = "https://openrouter.ai/api/v1"
# GEMINI_MODEL  = "google/gemini-2.5-flash"
# GEMINI_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

GEMINI_WORKERS   = 4
GEMINI_MAX_TOKENS = 1024

# ── Sampling ──
N_SAMPLES = 50
MAX_CONTENT_LENGTH = 12000

assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or OPENROUTER_API_KEY) env var before running"
print(f"Repo:   {REPO_NAME} ({REPO_LANGUAGE})")
print(f"Qwen:   {QWEN_MODEL_ID} @ {VLLM_URL}")
print(f"Gemini: {GEMINI_MODEL} @ {GEMINI_URL}")

## Collect & sample files

In [ ]:
REPOS_DIR   = ROOT / "repos"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

repo_dir = REPOS_DIR / REPO_NAME
assert repo_dir.is_dir(), f"Repo not found at {repo_dir} — place your repo there first"

config = get_language_config(REPO_LANGUAGE)

all_files = collect_files(
    str(repo_dir),
    extensions=config["extensions"],
    skip_dirs=config["skip_dirs"],
    skip_patterns=config.get("skip_patterns"),
)

random.seed(42)
files = random.sample(all_files, min(N_SAMPLES, len(all_files)))
print(f"Found {len(all_files)} files, sampled {len(files)}")

## Run both models

Inference helper with caching — same as the base eval notebook but parameterized for any endpoint.

In [ ]:
def run_model(files, prompt_template, default_result, *,
              model, base_url, api_key="", workers=4,
              max_tokens=1024, extra_body=None, extra_headers=None,
              cache_path=None, label="model"):
    """Run LLM inference on files with optional caching."""
    results = {}
    if cache_path and cache_path.exists():
        with open(cache_path) as f:
            results = json.load(f)

    remaining = [(fp, c) for fp, c in files if fp not in results]
    if not remaining:
        print(f"  [{label}] All {len(results)} files cached")
        return results

    print(f"  [{label}] {len(remaining)} files to process (workers={workers})")

    def process(args):
        fp, content = args
        prompt = prompt_template.replace(
            "FILEPATH_PLACEHOLDER", fp
        ).replace(
            "CONTENT_PLACEHOLDER", content[:MAX_CONTENT_LENGTH]
        )
        t0 = time.time()
        parsed = llm_extract(
            prompt, model=model, base_url=base_url, api_key=api_key,
            max_tokens=max_tokens, extra_body=extra_body,
            extra_headers=extra_headers,
        )
        elapsed = time.time() - t0
        if parsed is None or not isinstance(parsed, dict):
            return fp, dict(default_result), False, elapsed
        result = {k: parsed.get(k, v) for k, v in default_result.items()}
        return fp, result, True, elapsed

    done = 0
    total = len(remaining)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(process, item): item[0] for item in remaining}
        for future in as_completed(futures):
            fp, result, ok, elapsed = future.result()
            results[fp] = result
            done += 1
            status = "OK" if ok else "FAIL"
            print(f"    [{done}/{total}] {status} {fp} ({elapsed:.1f}s)")
            if done % 10 == 0 and cache_path:
                with open(cache_path, "w") as f:
                    json.dump(results, f, indent=2)

    if cache_path:
        with open(cache_path, "w") as f:
            json.dump(results, f, indent=2)
    return results

In [ ]:
repo_results_dir = RESULTS_DIR / REPO_NAME
repo_results_dir.mkdir(exist_ok=True)

print(f"=== Running Gemini ({GEMINI_MODEL}) ===")
gemini_results = run_model(
    files, config["prompt"], config["default_result"],
    model=GEMINI_MODEL,
    base_url=GEMINI_URL,
    api_key=GEMINI_API_KEY,
    workers=GEMINI_WORKERS,
    max_tokens=GEMINI_MAX_TOKENS,
    cache_path=repo_results_dir / "gemini_results.json",
    label="Gemini",
)

print(f"\n=== Running Qwen ({QWEN_MODEL_ID}) ===")
qwen_results = run_model(
    files, config["prompt"], config["default_result"],
    model=QWEN_MODEL_ID,
    base_url=VLLM_URL,
    api_key="",
    workers=QWEN_WORKERS,
    max_tokens=QWEN_MAX_TOKENS,
    extra_body=QWEN_EXTRA_BODY,
    cache_path=repo_results_dir / f"qwen_{QWEN_MODEL_ID}_results.json",
    label="Qwen",
)

print(f"\nGemini: {len(gemini_results)} files  |  Qwen: {len(qwen_results)} files")

## Compute metrics (Qwen vs Gemini)

In [ ]:
def _norm(item):
    if isinstance(item, dict):
        return json.dumps(item, sort_keys=True)
    return str(item).strip().lower()


def compute_metrics(gemini_results, qwen_results, default_result):
    common = sorted(set(gemini_results) & set(qwen_results))
    fields = list(default_result.keys())
    field_metrics = {}

    for field in fields:
        g_counts, q_counts, recalls, precisions, exact = [], [], [], [], 0
        for fp in common:
            g_val = gemini_results[fp].get(field, [])
            q_val = qwen_results[fp].get(field, [])
            if isinstance(g_val, list):
                g_set = {_norm(x) for x in g_val}
                q_set = {_norm(x) for x in q_val}
            else:
                g_set = {str(g_val)} if g_val else set()
                q_set = {str(q_val)} if q_val else set()

            g_counts.append(len(g_set))
            q_counts.append(len(q_set))
            recalls.append(len(g_set & q_set) / len(g_set) if g_set else (1.0 if not q_set else 0.0))
            precisions.append(len(g_set & q_set) / len(q_set) if q_set else (1.0 if not g_set else 0.0))
            if g_set == q_set:
                exact += 1

        n = len(common)
        r = sum(recalls) / n if n else 0
        p = sum(precisions) / n if n else 0
        field_metrics[field] = {
            "gemini_avg": round(sum(g_counts) / n, 2) if n else 0,
            "qwen_avg": round(sum(q_counts) / n, 2) if n else 0,
            "recall": round(r, 4),
            "precision": round(p, 4),
            "f1": round(2 * r * p / (r + p), 4) if (r + p) else 0,
            "exact_match": round(exact / n, 4) if n else 0,
        }

    return {"files_compared": len(common), "fields": field_metrics}

In [ ]:
metrics = compute_metrics(gemini_results, qwen_results, config["default_result"])

print(f"=== {REPO_NAME} ({REPO_LANGUAGE}) — {metrics['files_compared']} files ===")
print(f"  Teacher: {GEMINI_MODEL}")
print(f"  Student: {QWEN_MODEL_ID}")
print()
print(f"  {'Field':<22} {'G.avg':>6} {'Q.avg':>6}  {'Recall':>7} {'Prec':>7} {'F1':>6} {'Exact':>6}")
print(f"  {'-'*22} {'-'*6} {'-'*6}  {'-'*7} {'-'*7} {'-'*6} {'-'*6}")
for field, m in metrics["fields"].items():
    print(f"  {field:<22} {m['gemini_avg']:>6.1f} {m['qwen_avg']:>6.1f}"
          f"  {m['recall']:>7.1%} {m['precision']:>7.1%} {m['f1']:>6.3f} {m['exact_match']:>6.1%}")

## Per-file diff inspection

Browse files where the models disagree the most, to understand failure modes.

In [ ]:
def file_f1(gemini_res, qwen_res, fields):
    """Compute average F1 across fields for a single file."""
    f1s = []
    for field in fields:
        g_val = gemini_res.get(field, [])
        q_val = qwen_res.get(field, [])
        if isinstance(g_val, list):
            g_set = {_norm(x) for x in g_val}
            q_set = {_norm(x) for x in q_val}
        else:
            g_set = {str(g_val)} if g_val else set()
            q_set = {str(q_val)} if q_val else set()
        if not g_set and not q_set:
            f1s.append(1.0)
            continue
        r = len(g_set & q_set) / len(g_set) if g_set else 0.0
        p = len(g_set & q_set) / len(q_set) if q_set else 0.0
        f1s.append(2 * r * p / (r + p) if (r + p) else 0.0)
    return sum(f1s) / len(f1s) if f1s else 0.0


# Rank files by disagreement
common = sorted(set(gemini_results) & set(qwen_results))
fields = list(config["default_result"].keys())
scored = [(fp, file_f1(gemini_results[fp], qwen_results[fp], fields)) for fp in common]
scored.sort(key=lambda x: x[1])

N_SHOW = 5
print(f"=== Top {N_SHOW} worst-agreement files ===\n")
for fp, f1 in scored[:N_SHOW]:
    print(f"F1={f1:.3f}  {fp}")
    for field in fields:
        g = gemini_results[fp].get(field, [])
        q = qwen_results[fp].get(field, [])
        if g != q:
            print(f"  {field}:")
            print(f"    gemini: {g}")
            print(f"    qwen:   {q}")
    print()

## Save report

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
report_base = RESULTS_DIR / f"private_{REPO_NAME}_{QWEN_MODEL_ID}_{ts}"

# JSON
with open(str(report_base) + ".json", "w") as f:
    json.dump(metrics, f, indent=2)

# TXT
with open(str(report_base) + ".txt", "w") as f:
    f.write(f"Private Repo Eval — Dependency Analysis\n")
    f.write(f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Repo      : {REPO_NAME} ({REPO_LANGUAGE})\n")
    f.write(f"Teacher   : {GEMINI_MODEL}\n")
    f.write(f"Student   : {QWEN_MODEL_ID} (vLLM)\n\n")
    f.write(f"{'=' * 60}\n")
    f.write(f"FILES COMPARED: {metrics['files_compared']}\n")
    f.write(f"{'=' * 60}\n")
    f.write(f"  {'Field':<22} {'G.avg':>6} {'Q.avg':>6}  {'Recall':>7} {'Prec':>7} {'F1':>6} {'Exact':>6}\n")
    for field, m in metrics["fields"].items():
        f.write(f"  {field:<22} {m['gemini_avg']:>6.1f} {m['qwen_avg']:>6.1f}"
                f"  {m['recall']:>7.1%} {m['precision']:>7.1%} {m['f1']:>6.3f} {m['exact_match']:>6.1%}\n")

# Markdown
with open(str(report_base) + ".md", "w") as f:
    f.write(f"# Private Repo Eval: {REPO_NAME}\n\n")
    f.write(f"| | |\n|---|---|\n")
    f.write(f"| **Repo** | {REPO_NAME} ({REPO_LANGUAGE}) |\n")
    f.write(f"| **Teacher** | {GEMINI_MODEL} |\n")
    f.write(f"| **Student** | {QWEN_MODEL_ID} |\n")
    f.write(f"| **Files** | {metrics['files_compared']} |\n")
    f.write(f"| **Date** | {datetime.now().strftime('%Y-%m-%d')} |\n\n")
    f.write(f"| Field | G.avg | Q.avg | Recall | Precision | F1 | Exact Match |\n")
    f.write(f"|-------|:-----:|:-----:|:------:|:---------:|:--:|:-----------:|\n")
    for field, m in metrics["fields"].items():
        f.write(f"| **{field}** | {m['gemini_avg']:.1f} | {m['qwen_avg']:.1f}"
                f" | {m['recall']:.1%} | {m['precision']:.1%} | {m['f1']:.3f} | {m['exact_match']:.1%} |\n")

print(f"Saved: {report_base}.json")
print(f"Saved: {report_base}.txt")
print(f"Saved: {report_base}.md")